<a href="https://colab.research.google.com/github/LUMII-AILab/NLP_Course/blob/main/notebooks/LLM_as_a_service.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Atvērto LLM darbināšana, izmantojot OpenAI API

In [ ]:
# LU MII nodrošinātais API ir pilnībā savietojams ar OpenAI API
!pip install openai
from openai import OpenAI

In [2]:
models = {
    "google/gemma-3-27b-it": "https://gemma3.gauja.ailab.lv/v1/",
    "openai/gpt-oss-120b": "https://gpt120b.gauja.ailab.lv/v1/"
}

In [36]:
CURRENT_MODEL = "openai/gpt-oss-120b"

## Modeļu darbināšana instrukciju un teksta turpināšanas režīmos

In [4]:
my_prefix = "The following is a multiple choice question with answers A, B, C, D. The question and answers are in Latvian. Your task: answer the question by replying with 'A', 'B', 'C', or 'D', and nothing else."
my_prompt_1 = "Slavenākais latviešu komponists ir Raimonds \n A: Ešenvalds \n B: Vasks \n C: Pauls \n D: Vītols \n"
my_prompt_2 = "Slavenākais latviešu komponists ir Raimonds "
my_prompt_3 = "Kurš ir slavenākais latviešu komponists, kura vārds ir Raimonds?"

In [37]:
client = OpenAI(
    base_url = models[CURRENT_MODEL],
    api_key = "any_key"  # Pašlaik API Key netiek kontrolēts
)

In [8]:
# Kods modeļa darbināšanai čata jeb instrukciju sekošanas režīmā
response = client.chat.completions.create(
    model = CURRENT_MODEL,
    messages=[
        {"role": "system", "content": my_prefix},
        {"role": "user", "content": my_prompt_1},
    ],
    temperature=0.1,
    #max_tokens=1 # Neder GPT-OSS, jo tas vispirms ģenerē domāšanas tekstvienības
)
print(response.choices[0].message.content)

C


In [10]:
# Kods modeļa darbināšanai teksta turpināšanas režīmā
response = client.completions.create(
    prompt=my_prompt_2, # cf. my_prompt_3
    max_tokens=50,
    temperature=1.0,
    model=CURRENT_MODEL
)
print(response.choices[0].text)

Ērmanis, par to liecina arī "Lielā Mūzikas Balva 2024" nominācijas. Viņa mūzika ir atstājusi neizdzēšamu nospiedumu Latvijas


In [ ]:
# Pēc būtības tas pats, taču instrukciju režīmā
response = client.chat.completions.create(
    model = CURRENT_MODEL,
    messages=[
        {"role": "user", "content": my_prompt_3},
    ],
    temperature=1.0
)
print(response.choices[0].message.content)

Slavenākais latviešu komponists ar vārdu Raimonds ir, protams, **Raimonds Pauls**. Viņš ir ļoti pazīstams un ietekmīgs latviešu mūzikas kultūrā, pazīstams ar savām dziesmām, kas bieži vien ir melodiskas, emocionālas un tautiskas. 

Viņš ir komponējis vairāk nekā 1000 dziesmu un sadarbojies ar daudziem slaveniem Latvijas dziedātājiem. Viņa dziesmas ir kļuvušas par Latvijas kultūras mantojuma daļu.


## Vienkāršots RAG risinājums (Retrieval-Augmented Generation)

In [27]:
# Lietotāja jautājums
question = "Kāds ir sods par ātruma pārsniegšanu apdzīvotā vietā Latvijā par vairāk nekā 50 km/h?"

# Pieņemam, ka 'retrieval' daļa ir jau nostrādājusi
retrieved_chunks = [
    "Ātruma pārsniegšana apdzīvotā vietā līdz 10 km/st, < 7.5 t - sods ir brīdinājums [Ceļu satiksmes likuma 55.1. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 11-20 km/st, < 7.5 t - sods ir brīdinājums vai 4 NSV (20 EUR) [Ceļu satiksmes likuma 55.3. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 21-30 km/st, < 7.5 t - sods ir 8 NSV (40 EUR), punktu skaits: 1 [Ceļu satiksmes likuma 55.7. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 31-40 km/st, < 7.5 t - sods ir 16 NSV (80 EUR), punktu skaits: 2 [Ceļu satiksmes likuma 55.11. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 41-50 km/st, < 7.5 t - sods ir 32 NSV (160 EUR) līdz 44 NSV (220 EUR), punktu skaits: 3 [Ceļu satiksmes likuma 55.15. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 51-60 km/st, < 7.5 t - sods ir 48 NSV (240 EUR) līdz 64 NSV (320 EUR) un vadītāja tiesību atņemšana uz 3 mēn., punktu skaits: 4 [Ceļu satiksmes likuma 55.19. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 61-70 km/st, < 7.5 t - sods ir 144 NSV (720 EUR) līdz 192 NSV (960 EUR) un vadītāja tiesību atņemšana 9 mēn., punktu skaits: 5 [Ceļu satiksmes likuma 55.23. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā par vairāk nekā 70 km/st - sods ir 280 NSV (1400 EUR) līdz 400 NSV (2000 EUR) un vadītāja tiesību atņemšana 12–36 mēn., punktu skaits: 5 [Ceļu satiksmes likuma 55.26.1. pants]"
]

In [28]:
system_prompt = (
    "Tu esi jautājumu-atbilžu asistents par ceļu satiksmes pārkāpumiem Latvijā. "
    "Atbildi veido, izmantojot doto kontekstu. Atbildi pamato ar attiecīgo likuma pantu, ja tas ir zināms. "
    "Ja atbildi nav iespējams izsecināt no konteksta, tad atbildi, ka tev nav pietiekamas informācijas, lai atbildētu uz šo jautājumu. "
)

context = "\n\n".join(
    f"[{i+1}] {chunk}" for i, chunk in enumerate(retrieved_chunks)
)

user_prompt = f"Konteksts:\n{context}\n\nJautājums:\n{question}\n\nAtbilde:"

response = client.chat.completions.create(
    model=CURRENT_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0.5
)

answer = response.choices[0].message.content
print(answer)

Par ātruma pārsniegšanu apdzīvotā vietā Latvijā par vairāk nekā 50 km/h, ja transportlīdzekļa masa ir mazāka par 7.5 tonnām, sods ir no 48 NSV (240 EUR) līdz 64 NSV (320 EUR) un vadītāja tiesību atņemšana uz 3 mēnešiem, kā arī 4 punktu pieskaitīšana. (Ceļu satiksmes likuma 55.19. pants).


In [31]:
# Dialoga turpināšana
context = user_prompt + answer

question = "Vai ir vēl kādi ātruma pārsniegšanas limiti virs 50 km/h?"
#question = "Kas ir NSV? Izdomā atbildi, pat ja kontekstā nav pietiekamas informācijas."

user_prompt = f"Konteksts:\n{context}\n\nJautājums:\n{question}\n\nAtbilde:"

response = client.chat.completions.create(
    model=CURRENT_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0.5
)

answer = response.choices[0].message.content
print(answer)

Jā, ir vēl limiti virs 50 km/h:

*   **Ātruma pārsniegšana apdzīvotā vietā 51-60 km/st**, < 7.5 t - sods ir 48 NSV (240 EUR) līdz 64 NSV (320 EUR) un vadītāja tiesību atņemšana uz 3 mēn., punktu skaits: 4 [Ceļu satiksmes likuma 55.19. pants]
*   **Ātruma pārsniegšana apdzīvotā vietā 61-70 km/st**, < 7.5 t - sods ir 144 NSV (720 EUR) līdz 192 NSV (960 EUR) un vadītāja tiesību atņemšana 9 mēn., punktu skaits: 5 [Ceļu satiksmes likuma 55.23. pants]
*   **Ātruma pārsniegšana apdzīvotā vietā par vairāk nekā 70 km/st** - sods ir 280 NSV (1400 EUR) līdz 400 NSV (2000 EUR) un vadītāja tiesību atņemšana 12–36 mēn., punktu skaits: 5 [Ceļu satiksmes likuma 55.26.1. pants]



## Vienkāršots Tool-Call risinājums

In [49]:
# Pieejamie rīki un to apraksts
def get_weather(place):
    forecasts = {
        "Rīga": "Rīgā: Mākoņains, bez nokrišņiem. Vējš lēns līdz mērens, D/DR. Temperatūra +6…+11°C.",
        "Tallina": "Tallinā: Apmācies, iespējams neliels lietus. Vējš mērens no dienvidiem. Temperatūra +5…+10°C.",
        "Viļņa": "Viļņā: Mākoņains, iespējams īslaicīgas, spēcīgas lietusgāzes. Vējš lēns, dienvidrietumu. Temperatūra +10…+15°C."
    }
    return forecasts.get(place, f"Nav pieejama prognoze: {place}")

import random
def get_location():
    return random.choice(["Rīga", "Tallina", "Viļņa"])

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Funkcija atgriež šīs dienas laikapstākļu prognozi dotajai vietai.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "Vietas nosaukums. Šis nosaukums ir jāievieto precīzi, kāds tas tika iegūts. Nosaukumu nedrīkst izmainīt vai tulkot angliski."
                    }
                },
                "required": ["place"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_location",
            "description": "Funkcija atgriež lietotāja atrašanās vietas nosaukumu.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

In [50]:
# Palīgfunkcija domāšanas procesa izdrukām
import json
from IPython.display import display, Markdown

def print_message(msg):
    if not isinstance(msg, dict):
        msg = msg.model_dump()
    print(f"Role: {msg.get('role', 'N/A')}")
    if 'content' in msg and msg['content']:
        display(Markdown(msg['content']))
    else:
        print(json.dumps(msg, indent=4, ensure_ascii=False))
    print("-" * 20)

In [48]:
messages = [
    {"role": "user", "content": "Vai šodien ceļi būs slideni?"}
]

while True:
    response = client.chat.completions.create(
        model=CURRENT_MODEL, # Izvēlētajam modelim jānodrošina Thinking/Tool-Calling režīms
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.0,
    )

    assistant_msg = response.choices[0].message
    print_message(assistant_msg)

    messages.append(assistant_msg)

    if not assistant_msg.tool_calls:
        break

    for tool_call in assistant_msg.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")

        if function_name == "get_weather":
            place = arguments.get("place")
            content = get_weather(place)

        elif function_name == "get_location":
            content = get_location()

        else:
            content = "Unknown function: " + function_name

        tool_reply = {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": content,
        }

        print_message(tool_reply)
        messages.append(tool_reply)

Role: assistant
{
    "content": null,
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": [
        {
            "id": "chatcmpl-tool-b11d0d258a946fd3",
            "function": {
                "arguments": "{}",
                "name": "get_location"
            },
            "type": "function"
        }
    ],
    "reasoning": "The user asks in Latvian: \"Vai šodien ceļi būs slideni?\" meaning \"Will the roads be slippery today?\" We need to provide weather info for the user's location. We need to get location first via get_location, then get weather for that place. Then answer about road conditions (likely based on precipitation, temperature, etc.). Use functions.",
    "reasoning_content": "The user asks in Latvian: \"Vai šodien ceļi būs slideni?\" meaning \"Will the roads be slippery today?\" We need to provide weather info for the user's location. We need to get location first via get_location,

Viļņa

--------------------
Role: assistant
{
    "content": null,
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": [
        {
            "id": "chatcmpl-tool-90530db8db049c50",
            "function": {
                "arguments": "{\"place\": \"Vi\\u013c\\u0146a\"}",
                "name": "get_weather"
            },
            "type": "function"
        }
    ],
    "reasoning": "Now we have location \"Viļņa\". Likely a place in Latvia. Use get_weather for that place.",
    "reasoning_content": "Now we have location \"Viļņa\". Likely a place in Latvia. Use get_weather for that place."
}
--------------------
Role: tool


Viļņā: Mākoņains, iespējams īslaicīgas, spēcīgas lietusgāzes. Vējš lēns, dienvidrietumu. Temperatūra +10…+15°C.

--------------------
Role: assistant


Šodien Viļņā ir mākoņains laiks ar iespējamām īslaicīgām, spēcīgām lietusgāzēm. Temperatūra svārstās no +10 °C līdz +15 °C, un vējš ir lēns, dienvidrietumu virzienā. 

Ņemot vērā mitrumu no lietus un temperatūru, kas ir virs sasalšanas punktu, ceļi var būt slapji, bet ne tik bīstami kā, ja būtu sniegs vai ledus. Tomēr, ja lietus gāzes ir spēcīgas, uz ceļa virsmas var uzkrāties ūdens, kas var radīt slidenu slāni, īpaši uz ēnu vietām vai krustojumiem. 

**Ieteikumi:**
- Brauciet uzmanīgi, īpaši pie pārejas no sausa uz slapju virsmu.
- Palieliniet drošības attālumu un braukšanas ātrumu.
- Ja iespējams, izvairieties no neapdzīvotām vai maz apgaismotām ceļa daļām, kur var būt vairāk slapjuma.
- Pārbaudiet, vai jūsu transportlīdzekļa riepas ir labā stāvoklī un ar pietiekamu protektoru.

Kopumā ceļi būs slapji, bet ne īpaši bīstami, ja ievērosiet piesardzību. 🚗💧

--------------------
